In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from typing import Callable, Sequence

from popt.config import D0, D1, universe, sect
from popt.config import _1M, _3M, _6M, _1Y
fctr = ["SPY", "AGG", "GLD"]
from popt.alpha.modules.features import (
    momentum, 
    drawdown, 
    volatility, 
    volatility_downside, 
    sharpe_like, 
    skewness
)
from popt.alpha.modules.features import FeatureBuilder, FeatureView

In [3]:
rd = pd.read_parquet("../../../data/return/return_d.parquet").loc[D0:D1]
rf = pd.read_parquet("../../../data/return/ffr_d.parquet").reindex(rd.index)
rx = rd - rf.values

In [4]:
fb = FeatureBuilder(ret_d=rx, tickers=sect, factors=fctr, lookback=252, first_date=D0, final_date=D1)
fb.add_feature(name="mom_1m", regress=True, z_scale=True, lookback=_1M, callback=momentum)
fb.add_feature(name="mom_3m", regress=True, z_scale=True, lookback=_3M, callback=momentum)
fb.add_feature(name="mdd_1m", regress=True, z_scale=True, lookback=_1M, callback=drawdown)
fb.add_feature(name="mdd_3m", regress=True, z_scale=True, lookback=_3M, callback=drawdown)
fb.add_feature(name="vol_1m", regress=True, z_scale=True, lookback=_1M, callback=volatility)
fb.add_feature(name="vol_3m", regress=True, z_scale=True, lookback=_3M, callback=volatility)
fb.add_feature(name="vld_1m", regress=True, z_scale=True, lookback=_1M, callback=volatility_downside)
fb.add_feature(name="vld_3m", regress=True, z_scale=True, lookback=_3M, callback=volatility_downside)
fb.add_feature(name="shp_3m", regress=True, z_scale=True, lookback=_3M, callback=sharpe_like)
fb.add_feature(name="skw_3m", regress=True, z_scale=True, lookback=_3M, callback=skewness)
fb.consolidate()

In [ ]:
fv = FeatureView(fb, subset=None)
fv.add_mask(["XLK"], ["mom_1m"], exclude=False) 
fv.add_mask(["XLV"], ["mom_1m"], exclude=True)  
fv.apply_masking()
fv.features[-1, :, 6]

array([        nan,  0.16699112, -0.09921235, -0.98127505, -0.51060344,
       -0.4317537 ,  2.0370804 ,  1.02344707,  0.30027042])

In [ ]:
# NOTE Add to FeatureBuilder later on, not needed right now. 
#     def save_to_npz(self, file_path: str, verbose=False) -> None:
#         self.consolidate()
#         np.savez(
#             file=file_path,  # "../../../data/feature/sector.npz"
#             features=self.features,
#             names=self.names,
#             lookbacks=self.lookbacks,
#             timeline=self.timeline,
#             tickers=self.tickers,
#             factors=self.factors,
#         )
#         if verbose == True: print(f"stored {file_path}")

#         T = ret_d.shape[0]
#         N = len(tickers)
#         K = len(factors)
#         lookback_regression = lookback
#         rr = ret_d[tickers].values
#         xx = ret_d[factors].values
#         ee = rolling_regression(self.rr, self.xx, lookback)
#         timeline = ret_d.index                
#         tickers = tickers
#         factors = factors

#         F: int = None
#         features:   np.ndarray = None
#         __features: list[np.ndarray] = []
#         names: list[str] = []
#         lookbacks: list[str] = []
#         callbacks: list[Callable[..., np.ndarray]] = []

In [ ]:
# r, x = rx[sect].values, rx[fctr].values
# r_eps = rolling_target(r, x, lookback=252)

# mom_1m = standard_scale( rolling_feature(r_eps, _1M, momentum) )
# mom_3m = standard_scale( rolling_feature(r_eps, _3M, momentum) )
# mdd_1m = standard_scale( rolling_feature(r_eps, _1M, drawdown) )
# mdd_3m = standard_scale( rolling_feature(r_eps, _3M, drawdown) )
# vol_1m = standard_scale( rolling_feature(r_eps, _1M, volatility) )
# vol_3m = standard_scale( rolling_feature(r_eps, _3M, volatility) )
# vld_1m = standard_scale( rolling_feature(r_eps, _1M, volatility_downside) )
# vld_3m = standard_scale( rolling_feature(r_eps, _3M, volatility_downside) )
# shp_3m = standard_scale( rolling_feature(r_eps, _3M, sharpe_like) )
# skw_3m = standard_scale( rolling_feature(r_eps, _3M, skewness) )

# feature_set = [
#     ("MOM 1M", mom_1m, _1M),
#     ("MOM 3M", mom_3m, _3M),
#     ("MDD 1M", mdd_1m, _1M),
#     ("MDD 3M", mdd_3m, _3M),
#     ("VOL 1M", vol_1m, _1M),
#     ("VOL 3M", vol_3m, _3M),
#     ("VLD 1M", vld_1m, _1M),
#     ("VLD 3M", vld_3m, _3M),
#     ("SHP 3M", shp_3m, _3M),
#     ("SKW 3M", skw_3m, _3M),
# ]

# timeline = rx.index.to_numpy()
# assets = rx.columns.to_numpy()
# names = np.array([e[0] for e in feature_set])
# lookbacks = np.array([e[2] for e in feature_set])
# features = np.r_[*[e[1][None] for e in feature_set]]
# features.shape


In [ ]:
# NOTE potential interactions - stroner / weaker certain days
# Multiply by best features and see if any improvements

timeline = rx.index

day_of_wk = timeline.day_of_week
# day_of_mt = timeline.day
# day_of_yr = timeline.day_of_year

dow_cos = np.cos(2 * np.pi /   7 * day_of_wk)
dow_sin = np.sin(2 * np.pi /   7 * day_of_wk)
# dom_cos = np.cos(2 * np.pi /  31 * day_of_mt)
# dom_sin = np.sin(2 * np.pi /  31 * day_of_mt)
# doy_cos = np.cos(2 * np.pi / 365 * day_of_yr)
# doy_sin = np.sin(2 * np.pi / 365 * day_of_yr)


In [ ]:
def fit_model() -> np.ndarray:
    ...
    # add cyclif FE - seasons, day of year sin and cos

In [8]:
eps = r_eps[-1]
eps.std(axis=0)

np.float64(0.00448524182445094)

In [9]:
# Two forms of regression happning
# 1. Factor removal from returns - residual returns
# 2. Fittign the OLS model - Alpha model

# NOTE start by precomputing the residual returns at each time step
# targets and features in blocks of data
# preprocess all data and save down to npz,
# then run model fit loop on top of that




# %%
# NOTE Regress Sectors and International ETFs on market (potentially gold and bonds as well)
# Rank idiosyncratic returns to get assets that perform well orthogonally to the market
# Otherwise predictor may rank which ETF has biggest market component

# XLK - Technology
# XLV - Heathcare
# XLF - Financials
# XLY - Consumer Discretionary
# XLI - Industrials
# XLP - Consumer Staples
# XLE - Energy
# XLU - Utilities
# XLB - Materials

# IBB - Biotech
# IYR - Real Estate

# rd[[
#     "SPY", "AGG", "GLD",
#     "XLK", "XLV", "XLF", "XLY", "XLI", "XLP", "XLE", "XLU", "XLB",  # sectors
#     "IBB", "IYR",  # extra
#     # "EWJ", "EWG", "EWU", "EWA", "EWH", "EWS", "EWZ", "EWT", "EWY", "EWP", "EWW", "EWD", "EWL", "EWC",  # international
# ]].corr()


# r = B * rm + e
# e = r - B * rm
# rm = rx[["SPY"]].values
# r  = rx[sect].values
# Beta = r.T @ rm / (rm.T @ rm)
# eps = r - Beta.T * rm
# df_beta = pd.Series(Beta.ravel(), index=sect)
# df_eps = pd.DataFrame(eps, columns=sect)
# df_eps.corr()